# ExamEase AI — Student Stress Prediction Model

This notebook trains a regression model to predict a student's **self-reported exam stress score (1–10)** from structured survey information.

**Target:** `stress`  
**Project target:** R² ≥ 0.90 (to be attempted honestly using valid preprocessing, feature engineering, model comparison and cross-validation).

> Important: The current survey contains only 54 responses. A high R² on such a small dataset can be unstable. This notebook reports the real evaluation results instead of forcing the score upward.


## 1. What this notebook does

Pipeline:

`CSV → Cleaning → Feature Engineering → Train/Test Split → Model Comparison → Cross-Validation → Final Model → Save Model`

**Why:** We first convert the Google Forms responses into numerical features that ML algorithms can understand. Then we compare several regression models and select the model using cross-validation rather than choosing a model only because it looks good on one test split.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

print('Libraries imported successfully.')

## 2. Load the survey dataset

The notebook is inside `notebooks/` and the CSV is inside `dataset/`, so the normal project path is `../dataset/student_survey.csv`.

If your CSV has a different filename, change only `DATA_PATH`.

In [ ]:
DATA_PATH = Path('../dataset/student_survey.csv')

if not DATA_PATH.exists():
    # Fallback for running this notebook from the project root.
    DATA_PATH = Path('dataset/student_survey.csv')

df_raw = pd.read_csv(DATA_PATH)

print('Dataset shape:', df_raw.shape)
display(df_raw.head())

## 3. Clean the Google Forms columns

**Why:** Google Forms creates very long column names. We remove timestamp/name because they are not useful model inputs, and rename the remaining columns to simple names.

The student's name is deliberately excluded from the model.

In [ ]:
df = df_raw.copy()

# Remove identity/submission metadata.
df = df.drop(columns=[df.columns[0], df.columns[1]])

# Rename columns by their current position.
df.columns = [
    'stress',
    'stress_cause',
    'stress_time',
    'study_hours',
    'sleep_hours',
    'breaks',
    'planning_difficulty',
    'ai_interest',
    'support_needed',
    'stress_reduction'
]

df['stress_reduction'] = df['stress_reduction'].fillna('Not provided')

print('Clean shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())

## 4. Quick target analysis

The model predicts the student's stress score from 1 to 10.

`mean` = average stress.  
`median` = middle stress value.  
`R²` = how much variation in the target the model explains compared with a simple mean baseline.

In [ ]:
print('Average stress:', round(df['stress'].mean(), 2))
print('Median stress:', df['stress'].median())
print('\nStress distribution:')
print(df['stress'].value_counts().sort_index())

sns.countplot(data=df, x='stress')
plt.title('Exam Stress Score Distribution')
plt.xlabel('Stress Score (1–10)')
plt.ylabel('Number of Students')
plt.show()

## 5. Feature engineering

Survey answers such as `4–6 hours` are text, but ML models need numerical values. We convert them to representative numeric values.

We also convert multi-select questions into separate 0/1 indicator features. For example, if a student selected `Not enough time` and `Poor planning`, both corresponding features become 1.

`stress_reduction` is excluded because it is an open-ended response and is not a reliable structured input for this first model.

In [ ]:
def expand_multiselect(series, prefix):
    """Convert comma-separated multi-select answers into 0/1 columns."""
    lists = series.fillna('').astype(str).str.split(', ')
    choices = sorted({item for row in lists for item in row if item})
    result = pd.DataFrame(index=series.index)
    for i, choice in enumerate(choices):
        result[f'{prefix}_{i}'] = lists.apply(lambda items: int(choice in items))
    return result

features = pd.DataFrame(index=df.index)

study_map = {
    'Less than 1 hour': 0.5,
    '1–2 hours': 1.5,
    '2–4 hours': 3.0,
    '4–6 hours': 5.0,
    '6–8 hours': 7.0,
    'More than 8 hours': 9.0
}

sleep_map = {
    'Less than 4 hours': 3.0,
    '4–6 hours': 5.0,
    '6–8 hours': 7.0,
    'More than 8 hours': 9.0
}

break_map = {
    '❌ Almost never': 0,
    '🔴 Rarely': 1,
    '🟡 Sometimes': 2,
    '🟢 Yes, regularly': 3
}

ai_map = {
    '❌ No': 0,
    '👎 Probably not': 1,
    '🤔 Maybe': 2,
    '👍 Probably': 3,
    '🤩 Definitely': 4
}

features['study_hours'] = df['study_hours'].map(study_map)
features['sleep_hours'] = df['sleep_hours'].map(sleep_map)
features['breaks'] = df['breaks'].map(break_map)
features['planning_difficulty'] = pd.to_numeric(df['planning_difficulty'], errors='coerce')
features['ai_interest'] = df['ai_interest'].map(ai_map)

# Multi-select survey questions.
features = pd.concat([
    features,
    expand_multiselect(df['stress_cause'], 'cause'),
    expand_multiselect(df['stress_time'], 'time'),
    expand_multiselect(df['support_needed'], 'support')
], axis=1)

# Fill any unexpected unmapped values safely.
features = features.fillna(0)

X = features
y = df['stress'].astype(float)

print('Final feature matrix:', X.shape)
display(X.head())

## 6. Train/test split

We keep 20% of the data unseen during training.

**Why:** The test set gives us a more honest estimate of how the model performs on students it did not train on.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print('Training samples:', len(X_train))
print('Testing samples :', len(X_test))

## 7. Compare multiple regression models

We compare:

- Ridge Regression — simple linear baseline with regularization.
- Random Forest — ensemble of decision trees.
- Extra Trees — randomized tree ensemble.
- Gradient Boosting — sequential trees that improve previous errors.

We use **R², MAE and RMSE** together because one metric alone can be misleading.

In [ ]:
models = {
    'Ridge': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(
        n_estimators=300,
        max_depth=4,
        min_samples_leaf=2,
        random_state=42
    ),
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=2,
        random_state=42
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.03,
        min_samples_leaf=3,
        random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    results.append({
        'Model': name,
        'R2': r2_score(y_test, pred),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred) ** 0.5
    })
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
display(results_df.round(4))

## 8. 5-fold cross-validation

A single 20% test split is very small with only 54 students. Cross-validation gives several train/validation splits and helps us see whether a model is consistently useful.

The model with the highest **mean CV R²** is selected for the final model.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
    cv_results.append({
        'Model': name,
        'CV_R2_Mean': scores.mean(),
        'CV_R2_Std': scores.std()
    })

cv_df = pd.DataFrame(cv_results).sort_values('CV_R2_Mean', ascending=False)
display(cv_df.round(4))

## 9. Check the R² ≥ 0.90 target

This cell does **not** manipulate the score. It simply checks whether the honest cross-validation result reaches the project target.


In [ ]:
best_model_name = cv_df.iloc[0]['Model']
best_cv_r2 = cv_df.iloc[0]['CV_R2_Mean']

print('Selected model:', best_model_name)
print('Mean CV R²:', round(best_cv_r2, 4))

if best_cv_r2 >= 0.90:
    print('Target achieved: CV R² >= 0.90')
else:
    print('Target not achieved on the current 54-response dataset.')
    print('Do not force the metric. Collect more representative training data and retrain.')

## 10. Train the selected model on all available survey data

After evaluation, we retrain the selected model using all 54 responses. This creates the model artifact that can later be used by the ExamEase backend.

For production use, retrain again after collecting a much larger dataset.

In [ ]:
final_model = models[best_model_name]
final_model.fit(X, y)

MODEL_PATH = Path('../models/exam_ease_stress_model.joblib')
if not MODEL_PATH.parent.exists():
    MODEL_PATH = Path('models/exam_ease_stress_model.joblib')
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump({
    'model': final_model,
    'feature_columns': X.columns.tolist(),
    'best_model_name': best_model_name,
    'target': 'stress'
}, MODEL_PATH)

print('Model saved to:', MODEL_PATH)

## 11. Example prediction

This is only a demonstration using one existing row. In the application, the same feature-engineering function should be used on new student inputs before prediction.

In [ ]:
sample = X.iloc[[0]]
predicted_stress = final_model.predict(sample)[0]

print('Predicted stress:', round(float(predicted_stress), 2), '/ 10')
print('Actual stress   :', float(y.iloc[0]), '/ 10')

## 12. Next step for ExamEase AI

The current survey is a **prototype dataset**. The next ML improvement should be collecting substantially more structured responses and defining exactly which inputs will be available in the real app at prediction time.

Recommended pipeline:

`More student data → feature engineering → model tuning → cross-validation → external test set → deployment`

Do not use the student's name, timestamp, or free-text personal answer as model inputs.